Using DuckDB (ChatGPT Reco)

In [1]:
import pandas as pd
import duckdb

con = duckdb.connect("data.duckdb")

In [ ]:
con.execute("INSTALL excel")
con.execute("LOAD excel")

: 

In [3]:
con.execute("""
CREATE OR REPLACE TABLE abt_pt_jrny AS
SELECT * FROM read_csv_auto('data/A0008D - v_abt_pt_jrny (full).csv')
""")

con.execute("""
CREATE OR REPLACE TABLE abt_pt_ride AS
SELECT * FROM read_csv_auto('data/A0008D - v_abt_pt_ride (full).csv')
""")

con.execute("""
CREATE OR REPLACE TABLE pt_jrny AS
SELECT * FROM read_csv_auto('data/A0008D - v_pt_jrny_all (full).csv')
""")

con.execute("""
CREATE OR REPLACE TABLE pt_ride AS
SELECT * FROM read_csv_auto('data/A0008D - v_pt_ride_all (full).csv')
""")

con.execute("""
CREATE OR REPLACE TABLE bus_stop AS
SELECT * FROM read_csv_auto('data/A0008D - v_q_bus_stop (full).csv')
""")

con.execute("""
CREATE OR REPLACE TABLE vls_marker AS
SELECT * FROM read_csv_auto('data/A0008D - v_q_vls_marker (full).csv')
""")

mapping = pd.ExcelFile("data/Mapping.xlsx") # need to install openpyxl if don't have
for sheet in mapping.sheet_names:
    df = pd.read_excel(mapping, sheet_name=sheet)
    table_name = sheet.lower().replace(" ", "_")
    con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM df")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
tables_with_issues = {
    "svc_grade_id_num": pd.read_excel(mapping, sheet_name="SVC_GRADE_ID_NUM", skiprows=9),
    "pass_typ_cd": pd.read_excel(mapping, sheet_name="PASS_TYP_CD", skiprows=3)
}

for name, df in tables_with_issues.items():
    con.execute(f"""
        CREATE OR REPLACE TABLE {name} AS
        SELECT * FROM df
    """)

In [5]:
con.close()

Using SQLAlchemy (DSE3101 Week 4 Lect)

In [ ]:
# this alone took 5 mins...
abt_pt_jrny = pd.read_csv('data/A0008D - v_abt_pt_jrny (full).csv')

In [ ]:
import sqlalchemy2
engine = sqlalchemy.create_engine("sqlite:///../data/examples1.db"
connection = engine.connect()

In [ ]:
query = '''
CREATE TABLE IF NOT EXISTS flats (
    flat_id SERIAL PRIMARY KEY,
    commence_date INTEGER NOT NULL,
    remaining_lease FLOAT,
    town VARCHAR(100) NOT NULL,
    street_name VARCHAR(255) NOT NULL,
    block VARCHAR(10) NOT NULL,
    flat_type VARCHAR(50) NOT NULL,
    flat_model VARCHAR(50) NOT NULL,
    storey_range VARCHAR(50) NOT NULL,
    floor_area_sqm FLOAT NOT NULL
);
'''
pd.read_sql(query, engine)

Using PostgreSQL

In [ ]:
import psycopg2
import yaml

def load_config():
    with open('../config.yaml', 'r') as file:
        config = yaml.safe_load(file)
    return config

config = load_config()

database_config = config['database']
dbname = database_config['name']
user = database_config['user']
password = database_config['password']
host = database_config['host']

print(f'Connecting to {dbname} at {host} with user {user}')

In [ ]:
# Create a connection to PostgreSQL
try:
    conn = psycopg2.connect(dbname=dbname, user=user, password=password, host=host)
    print(f'Connected to PostgreSQL')
except psycopg2.Error as e:
    print(f'Error occurred: {e}')
    sys.exit(1) # Indicate error occurred during execution

# Create a cursor object
cursor = conn.cursor()

In [ ]:
cursor.execute("DROP TABLE IF EXISTS flats;")
cursor.execute("DROP TABLE IF EXISTS lease;")

In [ ]:
# Create the table if it doesn't already exist
create_table_query = """
CREATE TABLE IF NOT EXISTS flats (
    flat_id SERIAL PRIMARY KEY,
    commence_date INTEGER NOT NULL,
    remaining_lease FLOAT,
    town VARCHAR(100) NOT NULL,
    street_name VARCHAR(255) NOT NULL,
    block VARCHAR(10) NOT NULL,
    flat_type VARCHAR(50) NOT NULL,
    flat_model VARCHAR(50) NOT NULL,
    storey_range VARCHAR(50) NOT NULL,
    floor_area_sqm FLOAT NOT NULL
);

CREATE TABLE IF NOT EXISTS lease (
    lease_id SERIAL PRIMARY KEY,
    flat_id INTEGER NOT NULL,
    month TIMESTAMP NOT NULL,
    resale_price FLOAT NOT NULL,
    CONSTRAINT fk_flats FOREIGN KEY (flat_id) REFERENCES flats(flat_id)
    ON DELETE CASCADE
    ON UPDATE CASCADE
);
"""

try:
    cursor.execute(create_table_query)
    conn.commit()  # Commit the creation of the table
    print('Successfully created flats and lease tables')
except psycopg2.Error as e:
    print(f'Error occurred: {e}')
    conn.rollback() # Rollback transaction to prevent partial changes from being kept
    cursor.close() # Release resources
    conn.close() # Prevent connection leaks

In [ ]:
path1 = '../data/*.csv'
csvfiles = glob.glob(path1)
df1 = []

def validate_row(row):
    mandatory_fields = ['lease_commence_date', 'town', 'street_name', 'block', 'flat_type', 'flat_model', 'storey_range', 'floor_area_sqm', 'month', 
                        'resale_price']
    for field in mandatory_fields:
        if pd.isnull(row[field]):
            return False
    return True

for file in csvfiles:
    start1 = time.time()
    df1 = pd.read_csv(file)

    # Create a list to store values for batch insertion
    flat_values = []
    lease_values = []

    for index, row in df1.iterrows():
        if not validate_row(row):
            print(f'Skipping row {index} in file {file} due to missing mandatory information')
            continue

        # Prepare flat data for batch insert
        flat_values.append((row['lease_commence_date'], row['town'], row['street_name'], row['block'], row['flat_type'], row['flat_model'], 
                            row['storey_range'], row['floor_area_sqm'], row['remaining_lease']))

        # Prepare lease data for batch insert
        lease_values.append((row['town'], row['street_name'], row['month'], row['resale_price']))

    try:
        # Bulk insert into flats table
        insert_flats_query = """
            INSERT INTO flats (commence_date, town, street_name, block, flat_type, flat_model, storey_range, floor_area_sqm, remaining_lease)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
        """
        cursor.executemany(insert_flats_query, flat_values)

        # Commit after bulk insert
        conn.commit()

        # Retrieve flat_ids for the inserted flats in one query
        pairs_set = set()
        for row in lease_values:
            town = row[0]
            street_name = row[1]
            pairs_set.add((town, street_name))
        pairs = tuple(pairs_set) # A tuple of tuples

        cursor.execute("""
            SELECT flat_id, town, street_name FROM flats WHERE (town, street_name) IN %s
        """, (pairs,)) # Pass pairs as a single argument
        flat_ids = {f"{town}-{street_name}": flat_id for flat_id, town, street_name in cursor.fetchall()}

        # Now prepare lease data using the fetched flat_ids
        lease_values = [(flat_ids[f"{town}-{street_name}"], month, resale_price) for town, street_name, month, resale_price in lease_values]

        # Bulk insert into lease table
        insert_lease_query = """
            INSERT INTO lease (flat_id, month, resale_price)
            VALUES (%s, %s, %s)
        """
        cursor.executemany(insert_lease_query, lease_values)

        # Commit after bulk insert into lease table
        conn.commit()

        elapsed1 = time.time() - start1
        print(f'Successfully processed file {file} in {elapsed} seconds')
    
    except psycopg2.Error as e:
        print(f'Error processing file {file}: {e}')
        conn.rollback()
        continue

# Close the cursor and connection
cursor.close()
conn.close()

In [ ]:
# Close the cursor and connection
cursor.close()
conn.close()
print ('Closed connection')